In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    # Clear cache before loading model
    torch.cuda.empty_cache()

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
Number of GPUs: 1


In [3]:
# Initialize evaluation tracking
import pandas as pd
import json

repo_path = '/net/scratch2/smallyan/function_vectors_eval/'
eval_dir = os.path.join(repo_path, 'evaluation')
os.makedirs(eval_dir, exist_ok=True)

evaluation_results = []

def record_block_eval(block_id, block_desc, runnable, correct_impl, redundant, irrelevant, error_note=""):
    """Record evaluation for a code block"""
    evaluation_results.append({
        "Block_ID": block_id,
        "Description": block_desc,
        "Runnable": runnable,
        "Correct_Implementation": correct_impl,
        "Redundant": redundant,
        "Irrelevant": irrelevant,
        "Error_Note": error_note
    })
    return evaluation_results[-1]

print("Evaluation tracking initialized")
print(f"Evaluation directory: {eval_dir}")

Evaluation tracking initialized
Evaluation directory: /net/scratch2/smallyan/function_vectors_eval/evaluation


In [4]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 0
# ============================================
# Cell 0: Load autoreload extension
try:
    %load_ext autoreload
    %autoreload 2
    cell0_runnable = "Y"
    cell0_error = ""
except Exception as e:
    cell0_runnable = "N"
    cell0_error = str(e)

record_block_eval(
    "fv_demo.ipynb:cell-0", 
    "Load autoreload extension",
    cell0_runnable, "NA", "N", "N", cell0_error
)
print(f"Cell 0: Runnable={cell0_runnable}")

Cell 0: Runnable=Y


In [5]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 1
# ============================================
# Cell 1: Import libraries and utilities
try:
    # Change to the notebooks directory for correct relative imports
    os.chdir('/net/scratch2/smallyan/function_vectors_eval/notebooks')
    
    import os, re, json
    import torch, numpy as np

    import sys
    sys.path.append('..')
    torch.set_grad_enabled(False)

    from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
    from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
    from src.utils.model_utils import load_gpt_model_and_tokenizer
    from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
    from src.utils.eval_utils import decode_to_vocab, sentence_eval
    
    cell1_runnable = "Y"
    cell1_error = ""
except Exception as e:
    cell1_runnable = "N"
    cell1_error = str(e)

record_block_eval(
    "fv_demo.ipynb:cell-1", 
    "Import libraries and utility functions",
    cell1_runnable, "Y", "N", "N", cell1_error
)
print(f"Cell 1: Runnable={cell1_runnable}, Error={cell1_error}")

Cell 1: Runnable=Y, Error=


In [6]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 3 (Cell 2 is markdown)
# ============================================
# Cell 3: Load model & tokenizer
try:
    model_name = 'EleutherAI/gpt-j-6b'
    model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
    EDIT_LAYER = 9
    
    cell3_runnable = "Y"
    cell3_error = ""
    
    # Verify model is on CUDA
    print(f"Model loaded on device: {model.device}")
    print(f"Model config: {model_config}")
except Exception as e:
    cell3_runnable = "N"
    cell3_error = str(e)
    print(f"Error: {cell3_error}")

record_block_eval(
    "fv_demo.ipynb:cell-3", 
    "Load GPT-J model and tokenizer",
    cell3_runnable, "Y", "N", "N", cell3_error
)
print(f"Cell 3: Runnable={cell3_runnable}")

Loading:  EleutherAI/gpt-j-6b


Some weights of the model checkpoint at EleutherAI/gpt-j-6b were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

Model loaded on device: cuda:0
Model config: {'n_heads': 16, 'n_layers': 28, 'resid_dim': 4096, 'name_or_path': 'EleutherAI/gpt-j-6b', 'attn_hook_names': ['transformer.h.0.attn.out_proj', 'transformer.h.1.attn.out_proj', 'transformer.h.2.attn.out_proj', 'transformer.h.3.attn.out_proj', 'transformer.h.4.attn.out_proj', 'transformer.h.5.attn.out_proj', 'transformer.h.6.attn.out_proj', 'transformer.h.7.attn.out_proj', 'transformer.h.8.attn.out_proj', 'transformer.h.9.attn.out_proj', 'transformer.h.10.attn.out_proj', 'transformer.h.11.attn.out_proj', 'transformer.h.12.attn.out_proj', 'transformer.h.13.attn.out_proj', 'transformer.h.14.attn.out_proj', 'transformer.h.15.attn.out_proj', 'transformer.h.16.attn.out_proj', 'transformer.h.17.attn.out_proj', 'transformer.h.18.attn.out_proj', 'transformer.h.19.attn.out_proj', 'transformer.h.20.attn.out_proj', 'transformer.h.21.attn.out_proj', 'transformer.h.22.attn.out_proj', 'transformer.h.23.attn.out_proj', 'transformer.h.24.attn.out_proj', 'tran

In [7]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 5 (Cell 4 is markdown)
# ============================================
# Cell 5: Load dataset and compute task-conditioned mean activations
try:
    dataset = load_dataset('antonym', seed=0)
    print(f"Dataset loaded: {dataset}")
    
    # This may take a few minutes - reduce N_TRIALS for faster evaluation
    mean_activations = get_mean_head_activations(dataset, model, model_config, tokenizer)
    print(f"Mean activations shape: {mean_activations.shape}")
    
    cell5_runnable = "Y"
    cell5_error = ""
except Exception as e:
    cell5_runnable = "N"
    cell5_error = str(e)
    print(f"Error: {cell5_error}")

record_block_eval(
    "fv_demo.ipynb:cell-5", 
    "Load dataset and compute mean head activations",
    cell5_runnable, "Y", "N", "N", cell5_error
)
print(f"Cell 5: Runnable={cell5_runnable}")

Dataset loaded: {'train': ICLDataset({
	features: ['input', 'output'],
	num_rows: 1678
}), 'valid': ICLDataset({
	features: ['input', 'output'],
	num_rows: 216
}), 'test': ICLDataset({
	features: ['input', 'output'],
	num_rows: 504
})}


Mean activations shape: torch.Size([28, 16, 97, 256])
Cell 5: Runnable=Y


In [8]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 7 (Cell 6 is markdown)
# ============================================
# Cell 7: Compute function vector (FV)
try:
    FV, top_heads = compute_universal_function_vector(mean_activations, model, model_config, n_top_heads=10)
    print(f"Function vector shape: {FV.shape}")
    print(f"Top heads: {top_heads}")
    
    cell7_runnable = "Y"
    cell7_error = ""
except Exception as e:
    cell7_runnable = "N"
    cell7_error = str(e)
    print(f"Error: {cell7_error}")

record_block_eval(
    "fv_demo.ipynb:cell-7", 
    "Compute universal function vector",
    cell7_runnable, "Y", "N", "N", cell7_error
)
print(f"Cell 7: Runnable={cell7_runnable}")

Function vector shape: torch.Size([1, 4096])
Top heads: [(15, 5, 0.0587), (9, 14, 0.0584), (12, 10, 0.0526), (8, 1, 0.0445), (11, 0, 0.0445), (13, 13, 0.019), (8, 0, 0.0184), (14, 9, 0.016), (9, 2, 0.0127), (24, 6, 0.0113)]
Cell 7: Runnable=Y


In [9]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 9 (Cell 8 is markdown)
# ============================================
# Cell 9: Prompt Creation - ICL, Shuffled-Label, Zero-Shot
try:
    # Sample ICL example pairs, and a test word
    dataset = load_dataset('antonym')
    word_pairs = dataset['train'][:5]
    test_pair = dataset['test'][21]

    prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True)
    sentence = create_prompt(prompt_data)
    print("ICL prompt:\n", repr(sentence), '\n\n')

    shuffled_prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
    shuffled_sentence = create_prompt(shuffled_prompt_data)
    print("Shuffled ICL Prompt:\n", repr(shuffled_sentence), '\n\n')

    zeroshot_prompt_data = word_pairs_to_prompt_data({'input':[], 'output':[]}, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
    zeroshot_sentence = create_prompt(zeroshot_prompt_data)
    print("Zero-Shot Prompt:\n", repr(zeroshot_sentence))
    
    cell9_runnable = "Y"
    cell9_error = ""
except Exception as e:
    cell9_runnable = "N"
    cell9_error = str(e)
    print(f"Error: {cell9_error}")

record_block_eval(
    "fv_demo.ipynb:cell-9", 
    "Create ICL, Shuffled, and Zero-Shot prompts",
    cell9_runnable, "Y", "N", "N", cell9_error
)
print(f"\nCell 9: Runnable={cell9_runnable}")

ICL prompt:
 '<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 


Shuffled ICL Prompt:
 '<|endoftext|>Q: hardware\nA: democracy\n\nQ: fascism\nA: compatible\n\nQ: incompatible\nA: health\n\nQ: illness\nA: software\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 


Zero-Shot Prompt:
 '<|endoftext|>Q: increase\nA:'

Cell 9: Runnable=Y


In [10]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 12 (Cell 10, 11 are markdown)
# ============================================
# Cell 12: Clean ICL Prompt Evaluation
try:
    # Check model's ICL answer
    clean_logits = sentence_eval(sentence, [test_pair['output']], model, tokenizer, compute_nll=False)

    print("Input Sentence:", repr(sentence), '\n')
    print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
    print("ICL Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
    
    cell12_runnable = "Y"
    cell12_error = ""
except Exception as e:
    cell12_runnable = "N"
    cell12_error = str(e)
    print(f"Error: {cell12_error}")

record_block_eval(
    "fv_demo.ipynb:cell-12", 
    "Clean ICL Prompt Evaluation",
    cell12_runnable, "Y", "N", "N", cell12_error
)
print(f"Cell 12: Runnable={cell12_runnable}")

Input Sentence: '<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 

Input Query: 'increase', Target: 'decrease'



ICL Prompt Top K Vocab Probs:
 [(' decrease', 0.73675), (' reduce', 0.07769), (' increase', 0.03435), (' decline', 0.01574), (' decreased', 0.01037)] 

Cell 12: Runnable=Y


In [11]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 14 (Cell 13 is markdown)
# ============================================
# Cell 14: Corrupted ICL Prompt (Shuffled) + FV intervention
try:
    # Perform an intervention on the shuffled setting
    clean_logits, interv_logits = function_vector_intervention(shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

    print("Input Sentence:", repr(shuffled_sentence), '\n')
    print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
    print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
    print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))
    
    cell14_runnable = "Y"
    cell14_error = ""
except Exception as e:
    cell14_runnable = "N"
    cell14_error = str(e)
    print(f"Error: {cell14_error}")

record_block_eval(
    "fv_demo.ipynb:cell-14", 
    "Shuffled ICL Prompt + FV intervention",
    cell14_runnable, "Y", "N", "N", cell14_error
)
print(f"\nCell 14: Runnable={cell14_runnable}")

Input Sentence: '<|endoftext|>Q: hardware\nA: democracy\n\nQ: fascism\nA: compatible\n\nQ: incompatible\nA: health\n\nQ: illness\nA: software\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' decrease', 0.0277), (' increase', 0.0199), (' reduce', 0.00935), (' notice', 0.00611), (' improve', 0.0057)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' decrease', 0.67599), (' reduce', 0.05488), (' decline', 0.02134), (' increase', 0.01038), (' decreased', 0.00722)]

Cell 14: Runnable=Y


In [12]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 16 (Cell 15 is markdown)
# ============================================
# Cell 16: Zero-Shot Prompt + FV intervention
try:
    # Intervention on the zero-shot prompt
    clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

    print("Input Sentence:", repr(zeroshot_sentence), '\n')
    print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
    print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
    print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))
    
    cell16_runnable = "Y"
    cell16_error = ""
except Exception as e:
    cell16_runnable = "N"
    cell16_error = str(e)
    print(f"Error: {cell16_error}")

record_block_eval(
    "fv_demo.ipynb:cell-16", 
    "Zero-Shot Prompt + FV intervention",
    cell16_runnable, "Y", "N", "N", cell16_error
)
print(f"\nCell 16: Runnable={cell16_runnable}")

Input Sentence: '<|endoftext|>Q: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

Zero-Shot Top K Vocab Probs:
 [(' increase', 0.14925), (' yes', 0.02272), (' I', 0.02189), (' the', 0.0212), (' 1', 0.01418)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' decrease', 0.24879), (' increase', 0.17958), (' reduce', 0.03223), (' improve', 0.00924), ('\n', 0.00602)]

Cell 16: Runnable=Y


In [13]:
# ============================================
# EVALUATION: fv_demo.ipynb Cell 18 (Cell 17 is markdown)
# ============================================
# Cell 18: Natural Text Prompt + FV intervention
try:
    sentence = f"The word \"{test_pair['input']}\" means"
    co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)

    print("Input Sentence: ", repr(sentence))
    print("GPT-J:" , repr(tokenizer.decode(co.squeeze())))
    print("GPT-J+FV:", repr(tokenizer.decode(io.squeeze())), '\n')
    
    cell18_runnable = "Y"
    cell18_error = ""
except Exception as e:
    cell18_runnable = "N"
    cell18_error = str(e)
    print(f"Error: {cell18_error}")

record_block_eval(
    "fv_demo.ipynb:cell-18", 
    "Natural Text Prompt + FV intervention",
    cell18_runnable, "Y", "N", "N", cell18_error
)
print(f"Cell 18: Runnable={cell18_runnable}")

Input Sentence:  'The word "increase" means'
GPT-J: 'The word "increase" means "to make larger, to enlarge, to expand'
GPT-J+FV: 'The word "increase" means "decrease" in the Bible.\n' 

Cell 18: Runnable=Y
